# 📈 Notebook 03: Expense Data Analytics & Business Insights Dashboard

สมุดบันทึกสำหรับดึงข้อมูลค่าใช้จ่ายจริงที่ผ่านการอนุมัติแล้วจากฐานข้อมูล SQLite มาวิเคราะห์ทางธุรกิจ สรุปยอด และดูแนวโน้มค่าใช้จ่าย

## 🛠️ Step 0: โหลดไลบรารีและเชื่อมต่อฐานข้อมูล SQLite

In [ ]:
import os
import sys
import sqlite3
import pandas as pd
from IPython.display import display

# ปรับ Working Directory และ sys.path ให้อยู่ที่ Root
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

from src.core.db import get_db_connection

conn = get_db_connection()
print(f"✅ Connected to SQLite database from Root: {os.getcwd()}")

## 💰 Step 1: ภาพรวมค่าใช้จ่ายทั้งหมด (Executive Summary KPI)

In [ ]:
df_summary = pd.read_sql_query("""
    SELECT 
        COUNT(DISTINCT document_id) as total_documents,
        SUM(net_amount) as grand_total_spend,
        SUM(vat_amount) as total_input_vat,
        AVG(net_amount) as avg_spend_per_receipt
    FROM expense_receipts
""", conn)

print("📊 Executive Summary KPIs:")
display(df_summary)

## 🏪 Step 2: ร้านค้าที่มียอดใช้จ่ายสูงสุด (Top Spending Merchants)

In [ ]:
df_merchants = pd.read_sql_query("""
    SELECT 
        merchant_name,
        COUNT(*) as receipt_count,
        SUM(net_amount) as total_spent,
        SUM(vat_amount) as total_vat
    FROM expense_receipts
    GROUP BY merchant_name
    ORDER BY total_spent DESC
    LIMIT 10
""", conn)

print("🏆 Top 10 Merchants by Spending:")
display(df_merchants)

## 🛍️ Step 3: สินค้าและบริการที่มีการสั่งซื้อบ่อยที่สุด (Top Line Items)

In [ ]:
df_items = pd.read_sql_query("""
    SELECT 
        item_name,
        SUM(quantity) as total_qty,
        SUM(total_price) as total_amount
    FROM receipt_items
    GROUP BY item_name
    ORDER BY total_amount DESC
    LIMIT 10
""", conn)

print("🛍️ Top 10 Purchased Items:")
display(df_items)

conn.close()
print("\n✅ Database connection closed.")